# 손실 함수와 최대우도

정규분포와 베르누이 분포를 그린 뒤, 자주 쓰는 손실이 특정 분포의 최대우도추정(MLE)과 같은 최적화 문제임을 수식으로 연결한다.

표기 \(N(\mu,\sigma^2)\)에서 둘째 인자는 분산이다. \(N(2, 0.5)\)는 평균 2, 분산 0.5, 표준편차 \(\sqrt{0.5}\)이다.

재현을 위해 난수 시드는 42로 고정한다. NumPy만 계산에 쓰고, 그림은 Matplotlib과 Seaborn으로 저장한다.


In [2]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
np.random.seed(42)

from src.probability import plot_bernoulli_pmfs, plot_normal_pdfs, softmax_total_gap
from src.calculus import softmax

plot_normal_pdfs(ROOT / "outputs" / "normal_pdf.png")
plot_bernoulli_pmfs(ROOT / "outputs" / "bernoulli_pmf.png")

logits = np.random.randn(5)
probabilities = softmax(logits)
gap = abs(float(probabilities.sum()) - 1.0)
assert gap < 1e-6, gap
print("소프트맥스 입력", np.array2string(logits, precision=4))
print("소프트맥스 출력", np.array2string(probabilities, precision=6))
print("합", float(probabilities.sum()), "1과의 차이", gap)
print("시드 42 기본 벡터의 차이", softmax_total_gap())


소프트맥스 입력 [ 0.4967 -0.1383  0.6477  1.523  -0.2342]
소프트맥스 출력 [0.16764  0.08884  0.19496  0.467843 0.080717]
합 0.9999999999999998 1과의 차이 2.220446049250313e-16
시드 42 기본 벡터의 차이 2.220446049250313e-16


## 정규분포 PDF와 베르누이 PMF

정규분포의 밀도는

$$
\phi(x;\mu,\sigma^2)
=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).
$$

위 코드가 `outputs/normal_pdf.png`에 \(N(0,1)\)과 \(N(2,0.5)\)를 함께 그린다. 평균이 오른쪽으로 옮겨지고 분산이 1에서 0.5로 줄면 봉우리가 오른쪽으로 이동하면서 좁아진다.

베르누이 분포 \(B(p)\)는 두 점만 갖는다.

$$
P(K=1)=p,\qquad P(K=0)=1-p.
$$

`outputs/bernoulli_pmf.png`는 \(B(0.3)\)과 \(B(0.7)\)이다. 성공 확률이 0.5를 넘으면 질량이 \(k=1\) 쪽으로 기울어진다.

소프트맥스는 로짓 \(z\)를 범주 확률로 바꾼다.

$$
\mathrm{softmax}(z)_i=\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

분모로 나누므로 성분의 합은 1이고, 위 셀에서 \(|\sum_i p_i-1|<10^{-6}\)을 확인한다. 구현은 최댓값 \(m=\max_j z_j\)를 빼서 \(e^{z_i-m}\)을 쓰지만, 분자와 분모에 \(e^{-m}\)이 같이 들어가 값은 같다.


## MSE 최소화는 정규분포 가정에서의 MLE

입력 \(x_i\)가 주어졌을 때 관측이 조건부 정규분포를 따른다고 가정한다.

$$
y_i\mid x_i\sim N\big(\mu_i,\sigma^2\big),
\qquad
\mu_i=f_\theta(x_i).
$$

분산 \(\sigma^2\)는 \(\theta\)와 무관한 양수 상수로 둔다. 표본이 독립이면 우도는

$$
L(\theta)
=\prod_{i=1}^{n}
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(-\frac{(y_i-\mu_i)^2}{2\sigma^2}\right).
$$

로그를 취하면

$$
\ell(\theta)
=-\frac{n}{2}\log(2\pi\sigma^2)
-\frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i-\mu_i)^2.
$$

첫 항은 \(\theta\)를 포함하지 않는다. 둘째 항의 계수 \(1/(2\sigma^2)\)는 양수이므로, \(\ell(\theta)\)를 최대로 만드는 \(\theta\)는 다음을 최소로 만드는 \(\theta\)와 같다.

$$
\sum_{i=1}^{n}(y_i-f_\theta(x_i))^2
$$

평균제곱오차는 이 제곱합을 \(n\)으로 나눈 것이다.

$$
\mathrm{MSE}(\theta)=\frac{1}{n}\sum_{i=1}^{n}(y_i-f_\theta(x_i))^2.
$$

\(1/n\)도 양수 상수이므로

$$
\arg\min_\theta\ \mathrm{MSE}(\theta)
=\arg\max_\theta\ \ell(\theta).
$$

즉 가우시안 잡음 가정에서 평균을 신경망 예측으로 두면, MSE를 줄이는 일과 로그우도를 키우는 일은 같은 매개변수에서 끝난다. 분산을 함께 추정하면 로그우도에 \(\log\sigma^2\) 항이 남아 MSE와 완전히 같은 식은 아니지만, \(\sigma^2\)를 고정하면 둘의 최적 \(\theta\)는 같다.

다음 셀은 이 항등식을 숫자로 확인한다. \(\mu\)가 바뀌어도 \(\ell+\mathrm{SSE}/(2\sigma^2)\)는 상수이다.


In [3]:
# ℓ(θ) + SSE / (2 σ^2) 는 μ와 무관한 상수여야 한다.
sigma2 = 0.5
y = np.array([0.0, 1.2, -0.4, 0.7])
predictions = [
    np.array([0.0, 0.0, 0.0, 0.0]),
    np.array([0.2, 1.0, -0.5, 0.4]),
    y.copy(),
]
constant_term = -0.5 * len(y) * np.log(2.0 * np.pi * sigma2)
anchors = []
for mu in predictions:
    sse = float(np.sum((y - mu) ** 2))
    log_likelihood = constant_term - sse / (2.0 * sigma2)
    anchors.append(log_likelihood + sse / (2.0 * sigma2))
    print(f"SSE={sse:.6f}  loglik={log_likelihood:.6f}  loglik+SSE/(2σ²)={anchors[-1]:.6f}")
assert np.allclose(anchors, anchors[0])
print("로그우도에서 제곱합 항을 빼면 예측값과 무관한 상수가 남는다.")


SSE=2.090000  loglik=-4.379460  loglik+SSE/(2σ²)=-2.289460
SSE=0.180000  loglik=-2.469460  loglik+SSE/(2σ²)=-2.289460
SSE=0.000000  loglik=-2.289460  loglik+SSE/(2σ²)=-2.289460
로그우도에서 제곱합 항을 빼면 예측값과 무관한 상수가 남는다.


## Cross-Entropy 최소화는 베르누이·범주 분포 가정에서의 MLE

### 베르누이와 이진 교차 엔트로피

정답 \(y_i\in\{0,1\}\)이고, 모델이 성공 확률 \(\hat{y}_i\in(0,1)\)를 낸다고 하자.

$$
P(y_i\mid x_i)=\hat{y}_i^{y_i}(1-\hat{y}_i)^{1-y_i}.
$$

독립 표본의 로그우도는

$$
\ell(\theta)
=\sum_{i=1}^{n}\Big(y_i\log\hat{y}_i+(1-y_i)\log(1-\hat{y}_i)\Big).
$$

이진 교차 엔트로피는 이 식의 부호를 평균 낸 것이다.

$$
\mathrm{BCE}(\theta)
=-\frac{1}{n}\sum_{i=1}^{n}\Big(y_i\log\hat{y}_i+(1-y_i)\log(1-\hat{y}_i)\Big).
$$

따라서 \(\mathrm{BCE}\)를 최소화하는 \(\theta\)는 베르누이 로그우도를 최대화하는 \(\theta\)와 같다. 앞 노트북의 손실 \(L=-(y\log\hat{y}+(1-y)\log(1-\hat{y}))\)는 표본 하나짜리 BCE이다.

### 범주 분포와 교차 엔트로피

클래스가 \(K\)개이고 정답이 원-핫 벡터 \(y_i\)이면

$$
P(y_i\mid x_i)=\prod_{k=1}^{K} p_{i,k}^{y_{i,k}},
\qquad
p_i=\mathrm{softmax}(z_i).
$$

로그우도는

$$
\ell(\theta)=\sum_{i=1}^{n}\sum_{k=1}^{K} y_{i,k}\log p_{i,k}.
$$

범주 교차 엔트로피는

$$
\mathrm{CE}(\theta)
=-\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K} y_{i,k}\log p_{i,k}.
$$

부호와 양의 상수 \(1/n\)만 다르므로

$$
\arg\min_\theta\ \mathrm{CE}(\theta)
=\arg\max_\theta\ \ell(\theta).
$$

소프트맥스 출력은 음이 아니고 합이 1이라 범주 분포의 확률벡터로 쓸 수 있다. 위에서 합이 1에서 \(10^{-6}\)보다 덜 어긋남을 확인한 것은 이 확률 조건을 수치로 고정한 것이다.

정리하면, MSE는 조건부 정규분포의 평균을 맞추는 MLE이고, 교차 엔트로피는 베르누이 또는 범주 분포의 확률을 맞추는 MLE이다. 손실의 모양은 어떤 분포로 관측을 설명할지 고른 결과이다.
